# import libraries


In [ ]:
import argparse

import numpy as np
import pandas as pd
import torch
from datasets import Dataset, load_dataset
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)


In [ ]:
LABEL_COLUMNS = [
    "toxicity",
    "severe_toxicity",
    "obscene",
    "threat",
    "insult",
    "identity_attack",
    "sexual_explicit",
]

TRAIN_SAMPLE_SIZE = 100000
TEST_SAMPLE_SIZE = 15000
VALIDATION_SAMPLE_SIZE = 15000
LABEL_THRESHOLD = 0.5

MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 128
OUTPUT_DIR = "./moderation-model"
FINAL_MODEL_PATH = f"{OUTPUT_DIR}/final"
ONNX_OUTPUT_DIR = "./moderation-model-onnx"
QUANTIZED_MODEL_PATH = "./moderation-model-quantized"
BATCH_SIZE = 32

## Data preparation

In [ ]:
def prepare_split(dataset_split, n_sample: int, seed: int = 44) -> pd.DataFrame:
    df = dataset_split.shuffle(seed=seed).select(range(n_sample)).to_pandas()

    for col in LABEL_COLUMNS:
        df[col] = (df[col] >= LABEL_THRESHOLD).astype(int)

    df = df[df["text"].str.strip().str.len() > 5].reset_index(drop=True)

    return df[["text"] + LABEL_COLUMNS]


In [ ]:
def prepare_data() -> None:
    dataset = load_dataset("google/civil_comments")

    train_df = prepare_split(dataset["train"], TRAIN_SAMPLE_SIZE, seed=33)
    test_df = prepare_split(dataset["test"], TEST_SAMPLE_SIZE, seed=33)
    valid_df = prepare_split(dataset["validation"], VALIDATION_SAMPLE_SIZE, seed=33)

    train_df.to_csv("train.csv", index=False)
    test_df.to_csv("test.csv", index=False)
    valid_df.to_csv("valid.csv", index=False)

    print("train size:", len(train_df), "test size:", len(test_df), "valid size:", len(valid_df))
    print(train_df[LABEL_COLUMNS].mean().round(3))


## Training


In [ ]:
def load_tokenized_datasets(tokenizer) -> tuple[Dataset, Dataset]:
    train_df = pd.read_csv("train.csv")
    val_df = pd.read_csv("valid.csv")

    def to_hf_dataset(df: pd.DataFrame) -> Dataset:
        ds = Dataset.from_pandas(df)
        ds = ds.add_column("labels", df[LABEL_COLUMNS].astype("float").values.tolist())

        ds = ds.map(
            lambda batch: tokenizer(
                batch["text"],
                truncation=True,
                padding="max_length",
                max_length=MAX_LENGTH,
            ),
            batched=True,
        )
        return ds

    return to_hf_dataset(train_df), to_hf_dataset(val_df)


## Evaluation

In [ ]:
def compute_metrics(eval_pred) -> dict:
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))  # sigmoid
    preds = (probs >= 0.5).astype(int)

    return {
        "f1_micro": f1_score(labels, preds, average="micro", zero_division=0),
        "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
        "precision_micro": precision_score(labels, preds, average="micro", zero_division=0),
        "recall_micro": recall_score(labels, preds, average="micro", zero_division=0),
    }


# Train



In [ ]:
def train_model() -> None:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(LABEL_COLUMNS),
        problem_type="multi_label_classification",
    )

    train_ds, val_ds = load_tokenized_datasets(tokenizer)

    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        num_train_epochs=3,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="f1_micro",
        logging_steps=50,
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    trainer.save_model(FINAL_MODEL_PATH)
    tokenizer.save_pretrained(FINAL_MODEL_PATH)
    trainer.evaluate()



In [ ]:
def predict_batch(texts, tokenizer, model, device) -> np.ndarray:
    inputs = tokenizer(
        texts, truncation=True, padding=True, max_length=MAX_LENGTH, return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.sigmoid(logits).cpu().numpy()

    return (probs >= 0.5).astype(int)

In [ ]:
def evaluate_model() -> None:
    device = "cuda" if torch.cuda.is_available() else "cpu"

    tokenizer = AutoTokenizer.from_pretrained(FINAL_MODEL_PATH)
    model = AutoModelForSequenceClassification.from_pretrained(FINAL_MODEL_PATH).to(device)
    model.eval()

    test_df = pd.read_csv("test.csv")
    all_preds = []

    for i in range(0, len(test_df), BATCH_SIZE):
        batch_texts = test_df["text"].iloc[i : i + BATCH_SIZE].tolist()
        preds = predict_batch(batch_texts, tokenizer, model, device)
        all_preds.append(preds)

    all_preds = np.vstack(all_preds)
    true_labels = test_df[LABEL_COLUMNS].values

    print(classification_report(true_labels, all_preds, target_names=LABEL_COLUMNS, zero_division=0))

    sample_idx = np.random.choice(len(test_df), size=5, replace=False)
    for idx in sample_idx:
        predicted = [LABEL_COLUMNS[j] for j in range(len(LABEL_COLUMNS)) if all_preds[idx][j] == 1]
        actual = [LABEL_COLUMNS[j] for j in range(len(LABEL_COLUMNS)) if true_labels[idx][j] == 1]
        print(f"\nText: {test_df['text'].iloc[idx][:100]}...")
        print(f"  Predicted: {predicted or ['clean']}")
        print(f"  Actual:    {actual or ['clean']}")



## ONNX export

In [ ]:
def export_onnx() -> None:
    from optimum.onnxruntime import ORTModelForSequenceClassification

    ort_model = ORTModelForSequenceClassification.from_pretrained(FINAL_MODEL_PATH, export=True)
    tokenizer = AutoTokenizer.from_pretrained(FINAL_MODEL_PATH)

    ort_model.save_pretrained(ONNX_OUTPUT_DIR)
    tokenizer.save_pretrained(ONNX_OUTPUT_DIR)


## Quantization

In [ ]:
def quantize_model() -> None:
    from optimum.onnxruntime import ORTQuantizer
    from optimum.onnxruntime.configuration import AutoQuantizationConfig

    print("Applying INT8 dynamic quantization to ONNX model...")

    quantizer = ORTQuantizer.from_pretrained(ONNX_OUTPUT_DIR)
    qconfig = AutoQuantizationConfig.avx512_vnni(is_static=False, per_channel=False)
    quantizer.quantize(save_dir=QUANTIZED_MODEL_PATH, quantization_config=qconfig)

    tokenizer = AutoTokenizer.from_pretrained(ONNX_OUTPUT_DIR)
    tokenizer.save_pretrained(QUANTIZED_MODEL_PATH)

    print(f"Quantized model saved to: {QUANTIZED_MODEL_PATH}")


In [ ]:
"""STAGES = {
    "prepare_data": prepare_data,
    "train": train_model,
    "evaluate": evaluate_model,
    "export_onnx": export_onnx,
    "quantize": quantize_model,
}


DEFAULT_STAGE = "prepare_data"

def run_pipeline():
    parser = argparse.ArgumentParser(description="LLM fine-tuning pipeline")
    parser.add_argument("--stage", choices=STAGES.keys(), default=DEFAULT_STAGE)

    #
    args, _ = parser.parse_known_args()

    STAGES[args.stage]()


if __name__ == "__main__":
    run_pipeline()"""

'STAGES = {\n    "prepare_data": prepare_data,\n    "train": train_model,\n    "evaluate": evaluate_model,\n    "export_onnx": export_onnx,\n    "quantize": quantize_model,\n}\n\n\nDEFAULT_STAGE = "prepare_data"\n\ndef run_pipeline():\n    parser = argparse.ArgumentParser(description="LLM fine-tuning pipeline")\n    parser.add_argument("--stage", choices=STAGES.keys(), default=DEFAULT_STAGE)\n\n    # parse_known_args بدل parse_args عشان ياخد بس اللي يعرفه\n    # ويتجاهل أي آرجيومنتات زايدة يبعتها Jupyter/Colab تلقائيًا\n    args, _ = parser.parse_known_args()\n\n    STAGES[args.stage]()\n\n\nif __name__ == "__main__":\n    run_pipeline()'

In [53]:
prepare_data()
train_model()
evaluate_model()

train size: 99734 test size: 14959 valid size: 14942
toxicity           0.081
severe_toxicity    0.000
obscene            0.006
threat             0.003
insult             0.059
identity_attack    0.007
sexual_explicit    0.003
dtype: float64


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/99734 [00:00<?, ? examples/s]

Map:   0%|          | 0/14942 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,F1 Micro,F1 Macro,Precision Micro,Recall Micro
0.022004,0.042858,3,0.639477,0.398846,0.687167,0.597977


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

                 precision    recall  f1-score   support

       toxicity       0.71      0.64      0.67      1235
severe_toxicity       0.00      0.00      0.00         0
        obscene       0.58      0.64      0.61        75
         threat       0.44      0.24      0.31        33
         insult       0.70      0.66      0.68       908
identity_attack       0.58      0.28      0.38       110
sexual_explicit       0.46      0.31      0.37        35

      micro avg       0.69      0.62      0.65      2396
      macro avg       0.50      0.40      0.43      2396
   weighted avg       0.69      0.62      0.65      2396
    samples avg       0.05      0.05      0.05      2396


Text: I went back and saw that, but it was just a passing remark, not a diatribe like the ones you have be...
  Predicted: ['clean']
  Actual:    ['clean']

Text: there is no so called anything bud, Trump is the President, get over it....
  Predicted: ['clean']
  Actual:    ['clean']

Text: Barack Obama:  "Elec